In [1]:
import calendar
import pandas as pd
from datetime import date, timedelta
from sqlalchemy import create_engine
from pandas.tseries.offsets import BDay

engine = create_engine("sqlite:///c:\\ruby\\port_lite\\db\\development.sqlite3")
conlite = engine.connect()

engine = create_engine("mysql+pymysql://root:@localhost:3306/stock")
const = engine.connect()

pd.set_option('display.max_row',None)

data_path = "../data/"
csv_path = "\\Users\\User\\iCloudDrive\\"
box_path = "\\Users\\User\\Dropbox\\"
one_path = "\\Users\\User\\OneDrive\\Documents\\Data\\"

today = date.today()
yesterday = today - timedelta(days=1)
today, yesterday

(datetime.date(2023, 1, 15), datetime.date(2023, 1, 14))

In [2]:
# specify the number of business days
num_days = 1
# convert the timedelta object to a BusinessDay object
num_business_days = BDay(num_days)
yesterday = today - num_business_days
#yesterday = yesterday.date()
print(f'today: {today}')
print(f'yesterday: {yesterday}')

today: 2023-01-15
yesterday: 2023-01-13 00:00:00


In [3]:
yesterday = yesterday.date()
a_year_ago = yesterday - timedelta(days=365)
yesterday, a_year_ago

(datetime.date(2023, 1, 13), datetime.date(2022, 1, 13))

### Get past one quarter data

In [4]:
format_dict = {
    'ttl_qty':'{:,}',
    'from_price':'{:.2f}','to_price':'{:.2f}','diff':'{:.2f}',
    'max_price':'{:.2f}','min_price':'{:.2f}',
    'pct':'{:,.2f}%',  
}

In [5]:
sql = """
SELECT name
FROM buy 
ORDER BY name
"""
df = pd.read_sql(sql, const)

names = df['name'].values.tolist()
in_p = ", ".join(map(lambda name: "'%s'" % name, names))
in_p

"'ASP', 'BANPU', 'BCH', 'CPNREIT', 'DCC', 'DIF', 'GVREIT', 'IVL', 'JASIF', 'KCE', 'MAKRO', 'MCS', 'NER', 'ORI', 'PTTGC', 'RCL', 'SCC', 'SCCC', 'SENA', 'STA', 'SYNEX', 'TFFIF', 'TMT', 'WHAIR', 'WHART'"

In [6]:
#one_qtr_ago = yesterday - timedelta(days=4)
one_qtr_ago = date(2022, 12, 16)

sql = """
SELECT name, date, price, qty, maxp, minp
FROM price 
WHERE date >= '%s' AND name IN (%s) 
ORDER BY name, date"""
sql = sql % (one_qtr_ago, in_p)
print(sql)


SELECT name, date, price, qty, maxp, minp
FROM price 
WHERE date >= '2022-12-16' AND name IN ('ASP', 'BANPU', 'BCH', 'CPNREIT', 'DCC', 'DIF', 'GVREIT', 'IVL', 'JASIF', 'KCE', 'MAKRO', 'MCS', 'NER', 'ORI', 'PTTGC', 'RCL', 'SCC', 'SCCC', 'SENA', 'STA', 'SYNEX', 'TFFIF', 'TMT', 'WHAIR', 'WHART') 
ORDER BY name, date


In [7]:
df = pd.read_sql(sql, const)
df.sample(5)

,name,date,price,qty,maxp,minp
382,STA,2022-12-20,18.50,4526699,19.10,18.40
131,GVREIT,2023-01-03,9.15,174743,9.15,9.05
119,DIF,2023-01-13,13.50,9868681,13.50,13.30
116,DIF,2023-01-10,13.30,5999052,13.30,13.20
28,BANPU,2022-12-28,13.70,58996837,13.70,13.40


In [8]:
data_path = "../data/"
file_name = "Quarterly-Price-by-Name.csv"
output_file = data_path + file_name

df.set_index("name", inplace=True)
df.to_csv(output_file, header=None)

### After call ruby ruby\chat-fin.rb

In [9]:
file_name   = 'summary.csv'
input_file = data_path + file_name
df = pd.read_csv(input_file, sep=',', index_col=None)
df.sample(5).style.format(format_dict)

,name,from_date,to_date,days,from_price,to_price,diff,pct,spd,avg,max_price,min_price,ttl_qty,trend,nbr
66,JASIF,2022-12-22,2022-12-26,2,8.00,8.00,0.00,0.00%,0,0,8.10,7.95,"17,969,777",Downward,168
132,RCL,2022-12-30,2023-01-04,3,30.75,29.25,-1.50,-4.88%,-6,-2,31.50,29.25,"12,188,423",Downward,314
177,TFFIF,2022-12-19,2022-12-21,2,7.70,7.70,0.00,0.00%,0,0,7.75,7.60,"4,628,889",nan,425
18,BCH,2022-12-16,2022-12-16,0,19.80,19.80,0.00,0.00%,0,0,20.00,19.60,"18,104,197",Upward,42
91,MAKRO,2023-01-13,2023-01-13,0,42.50,42.50,0.00,0.00%,0,0,42.75,41.75,"17,001,855",Upward,221


In [10]:
df['from_date'] = pd.to_datetime(df['from_date'])
df['to_date'] = pd.to_datetime(df['to_date'])
df['fm_m_d'] = df['from_date'].dt.strftime('%m-%d')
df['to_m_d'] = df['to_date'].dt.strftime('%m-%d')

### Group analysis

In [11]:
df.groupby("trend").nunique()

,name,from_date,to_date,days,from_price,to_price,diff,pct,spd,avg,max_price,min_price,ttl_qty,nbr,fm_m_d,to_m_d
trend,,,,,,,,,,,,,,,,
Downward,25,20,20,5,76,76,16,30,10,5,74,74,96,96,20,20
Upward,25,20,20,12,77,74,25,47,14,6,81,75,100,100,20,20


In [12]:
cols = 'name fm_m_d to_m_d days from_price to_price pct spd avg max_price min_price ttl_qty trend'.split()
mask = df.name == 'ASP'
df[mask][cols].style.format(format_dict)

,name,fm_m_d,to_m_d,days,from_price,to_price,pct,spd,avg,max_price,min_price,ttl_qty,trend
0,ASP,12-16,12-16,0,2.96,2.96,0.00%,0,0,2.98,2.96,"1,313,820",nan
1,ASP,12-19,12-21,2,2.94,2.94,0.00%,0,0,2.98,2.92,"8,059,471",Downward
2,ASP,12-22,12-30,6,2.96,2.98,0.68%,1,0,2.98,2.92,"11,181,503",Upward
3,ASP,01-03,01-05,2,2.96,2.96,0.00%,0,0,2.98,2.94,"8,840,544",Downward
4,ASP,01-06,01-13,5,2.98,3.06,2.68%,4,0,3.10,2.96,"18,302,165",Upward


In [13]:
cols = 'name fm_m_d to_m_d days from_price to_price pct spd avg max_price min_price ttl_qty trend'.split()
yesterday = '2023-01-13'
pct = 3.00
df.query('to_date == @yesterday and pct >= @pct')[cols].style.format(format_dict)

,name,fm_m_d,to_m_d,days,from_price,to_price,pct,spd,avg,max_price,min_price,ttl_qty,trend
62,IVL,01-09,01-13,4,38.50,41.75,8.44%,13,3,42.25,37.75,"158,282,681",Upward
191,TMT,01-06,01-13,5,7.70,8.00,3.90%,6,1,8.05,7.55,"1,913,871",Upward


In [14]:
pct = -0.01
df.query('to_date == @yesterday and pct <= @pct')[cols].style.format(format_dict)

,name,fm_m_d,to_m_d,days,from_price,to_price,pct,spd,avg,max_price,min_price,ttl_qty,trend
125,PTTGC,01-12,01-13,1,50.25,50.00,-0.50%,-1,-1,50.75,49.25,"38,521,704",Downward
143,SCC,01-12,01-13,1,356.00,355.00,-0.28%,-1,-1,358.00,353.00,"6,180,835",Downward


In [15]:
# It returns a DataFrameGroupBy object
df_groupby_trend = df.groupby('trend')
type(df_groupby_trend)

pandas.core.groupby.generic.DataFrameGroupBy

In [16]:
# To get the number of groups
df_groupby_trend.ngroups

2

In [17]:
# Stat of each group
df_groupby_trend.size()

trend
Downward     96
Upward      100
dtype: int64

In [18]:
# To retrieve one of the created groups
df_up = df_groupby_trend.get_group('Upward')
df_up[cols].sample(5).style.format(format_dict)

,name,fm_m_d,to_m_d,days,from_price,to_price,pct,spd,avg,max_price,min_price,ttl_qty,trend
113,ORI,12-23,12-30,5,11.20,12.10,8.04%,9,1,12.10,10.80,"68,755,307",Upward
4,ASP,01-06,01-13,5,2.98,3.06,2.68%,4,0,3.10,2.96,"18,302,165",Upward
103,NER,01-06,01-06,0,6.30,6.30,0.00%,0,0,6.40,6.30,"5,698,387",Upward
163,SYNEX,12-16,12-16,0,15.90,15.90,0.00%,0,0,16.00,15.50,"1,775,117",Upward
43,DCC,01-12,01-12,0,2.84,2.84,0.00%,0,0,2.84,2.80,"5,025,330",Upward


In [19]:
# To retrieve one of the created groups
df_dw = df_groupby_trend.get_group('Downward')
df_dw[cols].sample(5).style.format(format_dict)

,name,fm_m_d,to_m_d,days,from_price,to_price,pct,spd,avg,max_price,min_price,ttl_qty,trend
206,WHART,01-12,01-12,0,10.70,10.70,0.00%,0,0,10.80,10.70,"1,551,924",Downward
6,BANPU,12-19,12-19,0,13.50,13.50,0.00%,0,0,13.70,13.50,"40,140,468",Downward
70,JASIF,01-13,01-13,0,8.20,8.20,0.00%,0,0,8.25,8.20,"6,798,932",Downward
176,TFFIF,12-16,12-16,0,7.70,7.70,0.00%,0,0,7.70,7.55,"1,520,602",Downward
14,BANPU,01-03,01-06,3,13.40,12.30,-8.21%,-11,-4,13.70,12.30,"854,388,284",Downward


In [20]:
df.groupby('trend').days.max()

trend
Downward     4
Upward      11
Name: days, dtype: int64

In [21]:
df[df['days'] == 11][cols].style.format(format_dict)

,name,fm_m_d,to_m_d,days,from_price,to_price,pct,spd,avg,max_price,min_price,ttl_qty,trend
147,SCCC,12-22,01-06,11,149.50,158.50,6.02%,18,1,159.00,148.50,"1,451,909",Upward


In [22]:
df.query('days == 4 and trend == "Downward"')[cols].style.format(format_dict)

,name,fm_m_d,to_m_d,days,from_price,to_price,pct,spd,avg,max_price,min_price,ttl_qty,trend
119,PTTGC,12-19,12-23,4,46.50,44.75,-3.76%,-7,-2,47.00,44.50,"52,679,289",Downward
152,SENA,12-20,12-26,4,3.94,3.90,-1.02%,-2,-1,3.96,3.90,"2,766,037",Downward
179,TFFIF,12-26,12-30,4,7.75,7.65,-1.29%,-2,-1,7.80,7.60,"7,610,655",Downward
190,TMT,12-30,01-05,4,7.80,7.65,-1.92%,-3,-1,7.95,7.60,"1,180,838",Downward
195,WHAIR,12-29,01-04,4,7.60,7.45,-1.97%,-3,-1,7.65,7.40,"2,468,511",Downward


In [28]:
df.groupby(['name', 'trend'])[['days']].agg('mean').reset_index()

,name,trend,days
0,ASP,Downward,2.000000
1,ASP,Upward,5.500000
2,BANPU,Downward,0.666667
3,BANPU,Upward,0.428571
4,BCH,Downward,0.250000
5,BCH,Upward,1.600000
6,CPNREIT,Downward,1.000000
7,CPNREIT,Upward,2.500000
8,DCC,Downward,0.000000
9,DCC,Upward,3.000000


In [30]:
# there is a function called .agg() and it allows us to specify multiple aggregation functions at once
df.groupby('trend').days.agg(['max', 'count', 'median', 'mean'])

,max,count,median,mean
trend,,,,
Downward,4,96,0.0,0.979167
Upward,11,100,1.0,2.110000
